In [1]:

# ================================================================
# JEEVAN LINK - ASSISTANCE MODEL TESTING
# ================================================================

import pandas as pd
import numpy as np
import joblib
import json

from pathlib import Path
from sklearn.metrics import (
    classification_report,
    multilabel_confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

print("=" * 70)
print("JEEVAN LINK ASSISTANCE MODEL TESTING")
print("=" * 70)

# ================================================================
# 1. LOAD TRAINED MODEL FILES
# ================================================================

MODEL_DIR = Path(
    r"C:\Users\ishit\jeevan_link_ml\Models\Assistance_Model"
)

model_path = MODEL_DIR / "assistance_model.pkl"
vectorizer_path = MODEL_DIR / "assistance_tfidf_vectorizer.pkl"
labels_path = MODEL_DIR / "assistance_label_names.pkl"
thresholds_path = MODEL_DIR / "assistance_thresholds.json"

for file in [
    model_path,
    vectorizer_path,
    labels_path,
    thresholds_path
]:
    if not file.exists():
        raise FileNotFoundError(f"Required model file missing: {file}")

assistance_model = joblib.load(model_path)
tfidf = joblib.load(vectorizer_path)
LABELS = joblib.load(labels_path)

with open(thresholds_path, "r") as f:
    thresholds = json.load(f)

print("\nMODEL FILES LOADED SUCCESSFULLY")
print("Assistance categories:", LABELS)

# ================================================================
# 2. DEFINE TEST SCENARIOS
# ================================================================

test_cases = [

    {
        "scenario": "Food requirement",
        "message": "Our family has no food and we are starving. Please send food supplies.",
        "expected": ["food"]
    },

    {
        "scenario": "Water requirement",
        "message": "We urgently need clean drinking water. There is no water available in our village.",
        "expected": ["water"]
    },

    {
        "scenario": "Shelter requirement",
        "message": "The flood destroyed our house. We have nowhere to stay and need emergency shelter.",
        "expected": ["shelter"]
    },

    {
        "scenario": "Medical help",
        "message": "My father is seriously injured and needs a doctor and immediate medical treatment.",
        "expected": ["medical_help"]
    },

    {
        "scenario": "Medical products",
        "message": "We urgently need medicines, bandages and medical supplies for injured people.",
        "expected": ["medical_products"]
    },

    {
        "scenario": "Search and rescue",
        "message": "Several people are trapped under the collapsed building. Please send a rescue team immediately.",
        "expected": ["search_and_rescue"]
    },

    {
        "scenario": "Transport requirement",
        "message": "We need an ambulance to transport injured people to the nearest hospital.",
        "expected": ["transport"]
    },

    {
        "scenario": "Food and water",
        "message": "Our village urgently needs food packets and clean drinking water after the cyclone.",
        "expected": ["food", "water"]
    },

    {
        "scenario": "Multiple assistance requirements",
        "message": "The flood destroyed our homes. We need food, drinking water, temporary shelter and medical assistance.",
        "expected": ["food", "water", "shelter", "medical_help"]
    },

    {
        "scenario": "Rescue and medical emergency",
        "message": "People are trapped in the damaged building and several injured victims need rescue and medical treatment.",
        "expected": ["search_and_rescue", "medical_help"]
    },

    {
        "scenario": "Irrelevant general message",
        "message": "Hello everyone. I hope you are doing well. Have a wonderful day.",
        "expected": []
    },

    {
        "scenario": "General information",
        "message": "The government announced a meeting about disaster awareness next week.",
        "expected": []
    },

    {
        "scenario": "Volunteer resource offer",
        "message": "I have a car and can deliver food and water to affected families.",
        "expected": []
    },

    {
        "scenario": "Road information",
        "message": "The main road is damaged after the earthquake and traffic is moving slowly.",
        "expected": []
    }
]

# ================================================================
# 3. PREDICTION FUNCTION
# ================================================================

def predict_assistance(message):

    features = tfidf.transform([message])

    probabilities = assistance_model.predict_proba(
        features
    )[0]

    predictions = []

    for i, label in enumerate(LABELS):

        probability = float(probabilities[i])

        threshold = float(thresholds[label])

        if probability >= threshold:
            predictions.append(label)

    return predictions, probabilities


# ================================================================
# 4. RUN ALL TEST SCENARIOS
# ================================================================

results = []

print("\nRUNNING SCENARIO TESTS...")
print("=" * 70)

for case in test_cases:

    message = case["message"]
    expected = case["expected"]

    predicted, probabilities = predict_assistance(message)

    correct = set(expected) == set(predicted)

    results.append({
        "scenario": case["scenario"],
        "message": message,
        "expected_labels": ", ".join(expected) if expected else "None",
        "predicted_labels": ", ".join(predicted) if predicted else "None",
        "exact_match": correct,
        "correct_labels": ", ".join(
            sorted(set(expected) & set(predicted))
        ),
        "missed_labels": ", ".join(
            sorted(set(expected) - set(predicted))
        ),
        "extra_labels": ", ".join(
            sorted(set(predicted) - set(expected))
        )
    })

    print("\nScenario:", case["scenario"])
    print("Expected:", expected if expected else "None")
    print("Predicted:", predicted if predicted else "None")
    print("Exact match:", correct)

# ================================================================
# 5. CREATE TEST RESULTS DATAFRAME
# ================================================================

results_df = pd.DataFrame(results)

total_tests = len(results_df)
exact_matches = int(results_df["exact_match"].sum())

exact_match_rate = exact_matches / total_tests

print("\n" + "=" * 70)
print("SCENARIO TEST SUMMARY")
print("=" * 70)

print("Total scenarios:", total_tests)
print("Exact matches:", exact_matches)
print("Incorrect matches:", total_tests - exact_matches)
print("Exact match rate:", round(exact_match_rate * 100, 2), "%")

# ================================================================
# 6. CALCULATE MULTI-LABEL METRICS
# ================================================================

Y_true = np.zeros((len(test_cases), len(LABELS)), dtype=int)
Y_pred = np.zeros((len(test_cases), len(LABELS)), dtype=int)

for row_index, case in enumerate(test_cases):

    for label in case["expected"]:
        Y_true[row_index, LABELS.index(label)] = 1

    predicted_labels = predict_assistance(
        case["message"]
    )[0]

    for label in predicted_labels:
        Y_pred[row_index, LABELS.index(label)] = 1

micro_precision = precision_score(
    Y_true, Y_pred,
    average="micro",
    zero_division=0
)

micro_recall = recall_score(
    Y_true, Y_pred,
    average="micro",
    zero_division=0
)

micro_f1 = f1_score(
    Y_true, Y_pred,
    average="micro",
    zero_division=0
)

macro_f1 = f1_score(
    Y_true, Y_pred,
    average="macro",
    zero_division=0
)

print("\nSCENARIO METRICS")
print("Micro Precision:", round(micro_precision, 4))
print("Micro Recall:", round(micro_recall, 4))
print("Micro F1:", round(micro_f1, 4))
print("Macro F1:", round(macro_f1, 4))

# ================================================================
# 7. PER-CATEGORY TEST RESULTS
# ================================================================

print("\nPER-CATEGORY RESULTS")

print(
    classification_report(
        Y_true,
        Y_pred,
        target_names=LABELS,
        zero_division=0
    )
)

category_results = pd.DataFrame(
    classification_report(
        Y_true,
        Y_pred,
        target_names=LABELS,
        output_dict=True,
        zero_division=0
    )
).transpose()

# ================================================================
# 8. SAVE RESULTS
# ================================================================

results_df.to_csv(
    MODEL_DIR / "assistance_scenario_test_results.csv",
    index=False
)

category_results.to_csv(
    MODEL_DIR / "assistance_scenario_category_metrics.csv"
)

summary = {
    "total_scenarios": total_tests,
    "exact_matches": exact_matches,
    "exact_match_rate": float(exact_match_rate),
    "micro_precision": float(micro_precision),
    "micro_recall": float(micro_recall),
    "micro_f1": float(micro_f1),
    "macro_f1": float(macro_f1)
}

with open(
    MODEL_DIR / "assistance_scenario_test_summary.json",
    "w"
) as f:
    json.dump(summary, f, indent=4)

print("\n" + "=" * 70)
print("TESTING COMPLETED")
print("=" * 70)

print("\nSaved files:")
print("1. assistance_scenario_test_results.csv")
print("2. assistance_scenario_category_metrics.csv")
print("3. assistance_scenario_test_summary.json")

print("\nSaved inside:")
print(MODEL_DIR)

# ================================================================
# 9. DISPLAY COMPLETE RESULTS
# ================================================================

display(
    results_df[
        [
            "scenario",
            "expected_labels",
            "predicted_labels",
            "exact_match",
            "missed_labels",
            "extra_labels"
        ]
    ]
)


JEEVAN LINK ASSISTANCE MODEL TESTING

MODEL FILES LOADED SUCCESSFULLY
Assistance categories: ['food', 'water', 'shelter', 'medical_help', 'medical_products', 'search_and_rescue', 'transport']

RUNNING SCENARIO TESTS...

Scenario: Food requirement
Expected: ['food']
Predicted: ['food', 'water']
Exact match: False

Scenario: Water requirement
Expected: ['water']
Predicted: ['food', 'water', 'transport']
Exact match: False

Scenario: Shelter requirement
Expected: ['shelter']
Predicted: ['shelter', 'search_and_rescue', 'transport']
Exact match: False

Scenario: Medical help
Expected: ['medical_help']
Predicted: ['medical_help']
Exact match: True

Scenario: Medical products
Expected: ['medical_products']
Predicted: ['medical_help', 'medical_products']
Exact match: False

Scenario: Search and rescue
Expected: ['search_and_rescue']
Predicted: ['search_and_rescue']
Exact match: True

Scenario: Transport requirement
Expected: ['transport']
Predicted: ['medical_help']
Exact match: False

Scenari

,scenario,expected_labels,predicted_labels,exact_match,missed_labels,extra_labels
0,Food requirement,food,"food, water",False,,water
1,Water requirement,water,"food, water, transport",False,,"food, transport"
2,Shelter requirement,shelter,"shelter, search_and_rescue, transport",False,,"search_and_rescue, transport"
3,Medical help,medical_help,medical_help,True,,
4,Medical products,medical_products,"medical_help, medical_products",False,,medical_help
5,Search and rescue,search_and_rescue,search_and_rescue,True,,
6,Transport requirement,transport,medical_help,False,transport,medical_help
7,Food and water,"food, water","food, water",True,,
8,Multiple assistance requirements,"food, water, shelter, medical_help","food, water, shelter, medical_help, medical_pr...",False,,"medical_products, transport"
9,Rescue and medical emergency,"search_and_rescue, medical_help","medical_help, search_and_rescue, transport",False,,transport


In [2]:

import pandas as pd
from pathlib import Path

MODEL_DIR = Path(
    r"C:\Users\ishit\jeevan_link_ml\Models\Assistance_Model"
)

FILE = MODEL_DIR / "assistance_test_predictions.csv"

df = pd.read_csv(FILE)

LABELS = [
    "food",
    "water",
    "shelter",
    "medical_help",
    "medical_products",
    "search_and_rescue",
    "transport"
]

error_records = []

for label in LABELS:

    actual_col = label + "_actual"
    predicted_col = label + "_predicted"
    probability_col = label + "_probability"

    false_positive = df[
        (df[actual_col] == 0) &
        (df[predicted_col] == 1)
    ].copy()

    false_negative = df[
        (df[actual_col] == 1) &
        (df[predicted_col] == 0)
    ].copy()

    print("\n" + "=" * 65)
    print("CATEGORY:", label)
    print("False positives:", len(false_positive))
    print("False negatives:", len(false_negative))

    for _, row in false_positive.iterrows():
        error_records.append({
            "category": label,
            "error_type": "False Positive",
            "message": row["message"],
            "probability": row[probability_col]
        })

    for _, row in false_negative.iterrows():
        error_records.append({
            "category": label,
            "error_type": "False Negative",
            "message": row["message"],
            "probability": row[probability_col]
        })

    print("\nExamples of false positives:")
    if len(false_positive):
        display(
            false_positive[
                ["message", probability_col]
            ].head(5)
        )
    else:
        print("None")

    print("\nExamples of false negatives:")
    if len(false_negative):
        display(
            false_negative[
                ["message", probability_col]
            ].head(5)
        )
    else:
        print("None")

errors_df = pd.DataFrame(error_records)

errors_df.to_csv(
    MODEL_DIR / "assistance_error_analysis.csv",
    index=False
)

print("\nERROR ANALYSIS SAVED")
print(MODEL_DIR / "assistance_error_analysis.csv")
print("Total errors:", len(errors_df))



CATEGORY: food
False positives: 20
False negatives: 27

Examples of false positives:


,message,food_probability
57,"we are in need of water, anti biotics, face ma...",0.748012
61,We need medicine for Hypertension and tampons/...,0.607634
197,"Good evening people of Boulevard 15 Octobre, w...",0.639048
206,Please come help us. We want to live but can't...,0.670426
232,There has been no authority since the earthqua...,0.943071



Examples of false negatives:


,message,food_probability
49,"we do not have water, our house is compromised...",0.393883
80,I need to let my family know that I am starvin...,0.461526
82,"I'm hungry, That is why I am sending this mess...",0.488695
150,"Hello, think about me and if you can help in t...",0.347912
169,"SOS, SOS. 27 month old child in difficulties a...",0.260229



CATEGORY: water
False positives: 14
False negatives: 8

Examples of false positives:


,message,water_probability
167,"URGENT, please help us, children and olders ar...",0.660311
232,There has been no authority since the earthqua...,0.962717
252,"I live in Pont rouge, we dont have tents or dr...",0.878835
265,"Thank you for your advice, however we do not h...",0.736138
307,We have a well that is dry.It got full again c...,0.851873


... (19 outputs left out)

,message,transport_probability
3,"SOS SOS, please provide police officers on the...",0.386243
20,"People don't have no food neither, they would ...",0.300558
26,we are in need of food tentes corvers water mo...,0.291195
39,"Clercine 24, impasse Cenor. . we need aide, we...",0.397842
42,The survivors of Pernier Fatima We have about ...,0.378090



Examples of false negatives:


,message,transport_probability
1,We are at Gressier we needs assistance right a...,0.199007
96,There's no one to send me money,0.082259
176,We are the group Women in Democracy. Leogane h...,0.260443
188,The centre is in the Legliz de Dye de la nouve...,0.061753
204,I'm a university student at the state universi...,0.166932



ERROR ANALYSIS SAVED
C:\Users\ishit\jeevan_link_ml\Models\Assistance_Model\assistance_error_analysis.csv
Total errors: 461
